# GPT-OSS-20B Baseline Benchmark (boto3)

This notebook deploys `openai/gpt-oss-20b` to a SageMaker endpoint using an **Inference Component**, then runs a **SageMaker AI Benchmark Job** against it to measure performance. The AI Benchmarking Service runs benchmarks (AIPerf) as SageMaker Training Jobs in your account, producing metrics like request latency, output token throughput, time-to-first-token (TTFT), and inter-token latency.

> **⚡ Already have a deployed endpoint?** Skip directly to **Step 7 — Setup benchmark client**. Steps 1–6 are only needed if you don't have an endpoint yet and want to deploy one to try this benchmarking feature.

**Workflow:**
1. *(Optional)* Download `openai/gpt-oss-20b` model weights and upload to S3
2. Initialize clients and helper functions
3. Configure instance and model settings
4. Deploy endpoint and Inference Component — **skip if you already have an endpoint**
5. Verify the deployment with a test inference
6. *(Optional)* Cleanup endpoint
7. **→ Start here if you have an existing endpoint** — set up the benchmark client
8. Create a workload config defining the benchmark traffic pattern
9. Submit a benchmark job targeting your endpoint
10. Poll for results
11. View results in S3 console
12. Display benchmark plots and metrics
13. Cleanup benchmark resources

**Prerequisites:**
- An existing SageMaker endpoint **or** `openai/gpt-oss-20b` model weights in S3 to deploy one
- An IAM execution role with SageMaker, S3, and IAM PassRole permissions
- Caller IAM identity with `sagemaker:*` permissions
- An S3 output bucket for benchmark results
- Sufficient GPU quota for `ml.g6.12xlarge` (only needed if deploying a new endpoint)

> **IAM note:** The execution role trust policy must allow `sagemaker.amazonaws.com` to assume it. The benchmark service runs as a SageMaker Training Job in your account using this role.

## Step 1 — Install dependencies

Upgrade boto3 and install the SageMaker Python SDK.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

## Step 2 — Initialize clients and helper functions

Sets up boto3 clients for SageMaker and SageMaker Runtime. Also defines helper functions to poll endpoint and inference component status without depending on the SageMaker Python SDK.

In [ ]:
import time
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")          # client to interact with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to interact with SageMaker Endpoints

### Helper functions

- `wait_for_endpoint()` — polls until the endpoint leaves the `Creating` state
- `wait_for_ic()` — polls until the inference component leaves the `Creating` state

In [ ]:
def get_execution_role():
    """Get the SageMaker execution role ARN.
    Tries multiple methods in order:
    1. SageMaker Studio metadata file (most reliable inside Studio)
    2. SageMaker describe_domain / describe_user_profile API
    3. STS caller identity fallback
    """
    import os, json

    # Method 1: Studio metadata file
    for path in [
        '/opt/ml/metadata/resource-metadata.json',
        '/etc/opt/ml/sagemaker-notebook-instance-config.json',
    ]:
        if os.path.exists(path):
            try:
                with open(path) as f:
                    meta = json.load(f)
                role = meta.get('ExecutionRoleArn') or meta.get('RoleArn')
                if role:
                    return role
            except Exception:
                pass

    # Method 2: SageMaker domain/user profile API
    try:
        sm_client = boto3.client('sagemaker')
        # Try to get domain execution role from Studio context
        domains = sm_client.list_domains()['Domains']
        if domains:
            domain_id = domains[0]['DomainId']
            domain = sm_client.describe_domain(DomainId=domain_id)
            role = domain.get('DefaultUserSettings', {}).get('ExecutionRole')
            if role:
                return role
    except Exception:
        pass

    # Method 3: STS fallback — derive from assumed-role ARN
    sts = boto3.client('sts')
    arn = sts.get_caller_identity()['Arn']
    if ':assumed-role/' in arn:
        parts = arn.split(':')
        account = parts[4]
        role_name = parts[5].split('/')[1]
        return f"arn:aws:iam::{account}:role/{role_name}"
    raise RuntimeError(f"Cannot derive execution role from ARN: {arn}. Set role manually.")


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    while status == "Creating":
        time.sleep(sleep_time)
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        clear_output(wait=True)
        progress += ind
        print(progress)
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    while status == "Creating":
        time.sleep(sleep_time)
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        clear_output(wait=True)
        progress += ind
        print(progress)
    print(f"IC: '{ic_name}', Status: '{status}'")

### Resolve execution role

Uses `sagemaker.get_execution_role()` to retrieve the IAM execution role attached to the current SageMaker Studio environment. This role is guaranteed to have the correct trust policy for `sagemaker.amazonaws.com`.

> Override `role` manually with a role ARN if running outside SageMaker Studio.

In [ ]:
# Overwrite with your role ARN if running outside of SageMaker Studio
# Use the role ARN that matches your SageMaker Studio domain execution role
role = get_execution_role()
print(f"Derived role: {role}")

# If the derived role doesn't exist, override manually:
# role = "arn:aws:iam::946952788839:role/AmazonSageMaker-ExecutionRole-20260420T113638"
print(f"Using role: {role}")

### Ensure role trust policy allows SageMaker

The execution role must have a trust relationship that allows `sagemaker.amazonaws.com` to assume it.
The cell below checks the current trust policy and adds the required statement if it is missing.
This is a no-op if the trust policy is already correct.

In [ ]:
import json
from urllib.parse import urlparse as _urlparse

iam = boto3.client('iam')
role_name = role.split('/')[-1]

# ── Ensure trust policy allows sagemaker.amazonaws.com ───────────────────────
trust = iam.get_role(RoleName=role_name)['Role']['AssumeRolePolicyDocument']
principals = [
    s.get('Principal', {}).get('Service', '')
    for s in trust.get('Statement', [])
]
flat = [p for item in principals for p in (item if isinstance(item, list) else [item])]

if 'sagemaker.amazonaws.com' in flat:
    print('✅ Trust policy already allows sagemaker.amazonaws.com')
else:
    trust['Statement'].append({
        'Effect': 'Allow',
        'Principal': {'Service': 'sagemaker.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    })
    iam.update_assume_role_policy(
        RoleName=role_name,
        PolicyDocument=json.dumps(trust)
    )
    print(f'✅ Added sagemaker.amazonaws.com to trust policy for role: {role_name}')

print(f'\n⚠️  Ensure the role has s3:GetObject and s3:ListBucket on your model bucket.')
print(f'   If CreateModel fails with S3 access error, attach AmazonS3ReadOnlyAccess')
print(f'   to role "{role_name}" in the IAM console, then re-run Step 4.')

## (Optional) Download openai/gpt-oss-20b and upload to S3

Skip this step if the model weights are already in S3.

This cell downloads `openai/gpt-oss-20b` from Hugging Face using `hf_transfer` for fast multi-part download, then syncs the weights to your S3 bucket. Update `MODEL_S3_URI` in Step 3 to match your bucket before running.

> **Note:** The model is ~40GB. Downloading inside SageMaker Studio on a fast instance takes ~5–10 minutes with `hf_transfer` enabled.

In [ ]:
# Uncomment and run this cell if you need to upload the model to S3 first.
#
# LOCAL_MODEL_DIR = "/tmp/gpt-oss-20b"
# S3_UPLOAD_URI   = "s3://your-bucket/models/gpt-oss-20b/"  # <-- replace with your S3 path
#
# # Install fast transfer library
# !pip install -q "huggingface_hub[hf_transfer]"
#
# # Download model weights from Hugging Face (excludes GGUF and legacy .bin files)
# import os
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# !huggingface-cli download openai/gpt-oss-20b \
#     --local-dir {LOCAL_MODEL_DIR} \
#     --exclude '*.gguf' '*.bin'
#
# # Sync to S3
# !aws s3 sync {LOCAL_MODEL_DIR} {S3_UPLOAD_URI} --no-progress
# print(f"Model uploaded to {S3_UPLOAD_URI}")

## Step 3 — Configure container and instance

Defines the instance type, model S3 path, and unique resource names for this run.

| Setting | Value | Notes |
|---|---|---|
| Instance | `ml.g6.12xlarge` | 4×NVIDIA L40S GPUs |
| Model | `openai/gpt-oss-20b` | Loaded from S3 via vLLM container |
| Timeout | 900s | Allows time for model weight loading |

In [ ]:
instance = {"type": "ml.g6.12xlarge", "num_gpu": 4}
MODEL_S3_URI = "s3://your-bucket/models/gpt-oss-20b/"  # <-- replace with your S3 path
model_name = f"oss20b-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 900
variant_name = "v1"

## Step 4 — Deploy endpoint and inference component

Configures the vLLM container and creates the SageMaker model, endpoint config, and endpoint. Then attaches an **Inference Component (IC)** to the endpoint.

Key vLLM container environment variables:

| Variable | Value | Description |
|---|---|---|
| `SM_VLLM_MODEL` | `/opt/ml/model` | Path to model weights inside the container |
| `SM_VLLM_TENSOR_PARALLEL_SIZE` | `num_gpu` | Splits the model across all available GPUs |
| `SM_VLLM_MAX_NUM_SEQS` | `32` | Maximum concurrent sequences |
| `SM_VLLM_MAX_MODEL_LEN` | `8192` | Max context length — limits KV cache to fit in GPU memory |

Model weights are loaded from S3 using `ModelDataSource` with `S3Prefix` — SageMaker streams the weights directly into the container at startup.

Inference Components allow multiple model copies to share the same endpoint and enable fine-grained resource allocation. The benchmark job targets the IC directly via `InferenceComponents` in `BenchmarkTarget`.

In [ ]:
VLLM_IMAGE_URI = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.16.0-gpu-py312-cu129-ubuntu22.04-sagemaker-v1.1"

env = {
    "SM_VLLM_MODEL":                "/opt/ml/model",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(instance["num_gpu"]),
    "SM_VLLM_MAX_NUM_SEQS":         "32",
    "SM_VLLM_MAX_MODEL_LEN":        "16384",
    "SM_VLLM_ENFORCE_EAGER":        "true",
}

create_model_response = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": VLLM_IMAGE_URI,
        "Environment": env,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri":           MODEL_S3_URI,
                "S3DataType":      "S3Prefix",
                "CompressionType": "None",
            }
        },
    },
)
print(f"✅ Model created: {create_model_response['ModelArn']}")

create_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
)
print(f"✅ Endpoint config created: {create_endpoint_config_response['EndpointConfigArn']}")

create_endpoint_response = sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)
print(f"✅ Endpoint creation started: {create_endpoint_response['EndpointArn']}")
wait_for_endpoint(endpoint_name)

In [ ]:
ic_name = f"ic-{model_name}"


In [ ]:
ic_name = f"ic-{model_name}"

min_memory_required_in_mb = 40*1024  # 40GB for gpt-oss-20b
number_of_accelerator_devices_required = instance["num_gpu"]

create_ic_response = sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": timeout,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": min_memory_required_in_mb,
            "NumberOfAcceleratorDevicesRequired": number_of_accelerator_devices_required,
        },
    },
    RuntimeConfig={
        "CopyCount": 1,
    },
)
print(f"Inference Component created: {create_ic_response['InferenceComponentArn']}")
wait_for_ic(ic_name)

## Step 5 — Test inference

Sends a test request to verify the endpoint is responding correctly before running the benchmark. The endpoint must be in `InService` status and support the OpenAI-compatible chat/completions API — the benchmarking service uses a transparent proxy that translates OpenAI requests to SageMaker `InvokeEndpoint` calls.

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "Solve this problem step by step: What is 15% of 240?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 InferenceComponentName=ic_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"]
print(f'-----------------------\n{usage}')

## Step 6 — Cleanup endpoint (optional)

Uncomment to delete the inference component, endpoint, endpoint config, and model when done. Skip this if you want to keep the endpoint running for the benchmark job below.

In [ ]:
#_ = sm.delete_inference_component(InferenceComponentName=ic_name)

In [ ]:
#_ = sm.delete_endpoint(EndpointName=endpoint_name)
#_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
#_ = sm.delete_model(ModelName=model_name)

## Step 7 — Setup benchmark client

> **⚡ Start here if you already have a deployed endpoint.** Set `endpoint_name` and `ic_name` to your existing endpoint and inference component names, then run from this step onward.

Initializes a SageMaker client to access the AI Benchmarking APIs.

In [ ]:
import boto3
import json
from datetime import datetime
import uuid

# Re-use the region from the session, or override here
REGION  = region  # set above from boto_session.region_name; override if needed
session = boto3.Session(region_name=REGION)

# Create SageMaker client
client = session.client("sagemaker")

# Auto-generate unique names on every run
run_id      = datetime.now().strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
config_name = f"bmk-config-{run_id}"
job_name    = f"bmk-job-{run_id}"

print(f"Config name : {config_name}")
print(f"Job name    : {job_name}")

## Step 8 — Create workload config

Defines the benchmark traffic pattern using the **ShareGPT public dataset** at concurrency=10.

**Workload parameters:**
| Parameter | Value | Description |
|---|---|---|
| `public_dataset` | `sharegpt` | Use the ShareGPT public dataset for realistic prompts |
| `concurrency` | `10` | Number of simultaneous requests |
| `request_count` | `300` | Total requests to send during the benchmark |
| `prompt_input_tokens_mean` | `500` | Average input token length |
| `output_tokens_mean` | `500` | Average output token length |
| `extra_inputs` | `ignore_eos:true` | Forces the model to generate the full requested output length |

> **Note:** `prompt_input_tokens_mean` / `prompt_input_tokens_stddev` are only supported with synthetic datasets. Public and custom datasets supply their own input prompts.

In [ ]:
workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "public_dataset": "sharegpt",
        "prompt_input_tokens_mean": 500,
        "prompt_input_tokens_stddev": 10,
        "output_tokens_mean": 200,
        "output_tokens_stddev": 10,
        "extra_inputs": "ignore_eos:true",
        "concurrency": 10,
        "request_count": 300,
    },
}

response = client.create_ai_workload_config(
    AIWorkloadConfigName=config_name,
    AIWorkloadConfigs={
        "WorkloadSpec": {"Inline": json.dumps(workload_spec)}
    },
)

print(response["AIWorkloadConfigArn"])

## Step 9 — Submit benchmark job

Submits a benchmark job targeting the deployed inference component. The service runs AIPerf as a SageMaker Training Job in your account and writes results to S3.

The `BenchmarkTarget` specifies both the endpoint and the inference component — the `InferenceComponents` field is optional and only needed for IC-based endpoints.

In [ ]:
job_response = client.create_ai_benchmark_job(
    AIBenchmarkJobName=job_name,
    BenchmarkTarget={
        "Endpoint": {
            "Identifier": endpoint_name,
            "InferenceComponents": [  # optional, for IC-endpoints
                {"Identifier": ic_name},
            ]
        }
    },
    OutputConfig={
        "S3OutputLocation": "s3://your-bucket/benchmark-output/"  # <-- replace with your S3 output path
    },
    AIWorkloadConfigIdentifier=config_name,
    RoleArn="arn:aws:iam::XXXXXXXXXXXX:role/SageMaker-Benchmark-Role",  # <-- replace with your role ARN
)

print(job_response["AIBenchmarkJobArn"])

## Step 10 — Poll for benchmark results

Polls every 30 seconds until the benchmark job completes. Results are written to the S3 output location and include:
- `profile_export_aiperf.json` — aggregated metrics (latency percentiles, throughput, TTFT, ITL)
- `profile_export_aiperf.csv` — same metrics in CSV format
- `profile_export.jsonl` — per-request raw data
- `plots/` — TTFT timeline and over-time plots

In [ ]:
import time

while True:
    response = client.describe_ai_benchmark_job(AIBenchmarkJobName=job_name)
    status = response["AIBenchmarkJobStatus"]
    print(f"Status: {status}")

    if status in ("Completed", "Failed", "Stopped"):
        break

    time.sleep(30)

if status == "Completed":
    print("Benchmark completed successfully!")
    print(f"Results at: {response['OutputConfig']['S3OutputLocation']}")
elif status == "Failed":
    print(f"Benchmark failed: {response.get('FailureReason', 'unknown')}")

## Step 11 — View results in S3 console

Generates a clickable link to browse the benchmark output folder directly in the AWS S3 console.

The results folder contains:

| File | Description |
|---|---|
| `profile_export_aiperf.json` | Aggregated metrics — latency percentiles, throughput, TTFT, ITL |
| `profile_export_aiperf.csv` | Same metrics in CSV format |
| `profile_export.jsonl` | Per-request raw data |
| `plots/` | TTFT timeline and over-time charts |

In [ ]:
from urllib.parse import urlparse, quote
from IPython.display import display, Markdown

s3_output = response['OutputConfig']['S3OutputLocation']
parsed = urlparse(s3_output)
bucket = parsed.netloc
prefix = parsed.path.lstrip('/')

# Build the S3 console URL
console_url = (
    f"https://s3.console.aws.amazon.com/s3/buckets/{bucket}"
    f"?region={region}&prefix={quote(prefix, safe='/')}"
)

display(Markdown(f"### 📂 Benchmark Results\n"
                 f"**S3 path:** `{s3_output}`\n\n"
                 f"**[Click here to view results in S3 console]({console_url})**"))

## Step 12 — Display benchmark plots

Downloads the PNG charts from S3 and renders them inline. The benchmark produces two plots:
- **`ttft_timeline.png`** — time-to-first-token for each individual request in order
- **`ttft_over_time.png`** — TTFT aggregated across the duration of the run

In [ ]:
import boto3
import io
import json
import tarfile
import matplotlib.pyplot as plt
import numpy as np
from urllib.parse import urlparse, quote
from IPython.display import display, Image, Markdown

s3_client = boto3.client('s3')
parsed  = urlparse(s3_output)
bucket  = parsed.netloc
prefix  = parsed.path.lstrip('/')

# ── Download and extract output.tar.gz ───────────────────────────────────────
tar_key = f"{prefix}output/output.tar.gz"
print(f"Downloading: s3://{bucket}/{tar_key}")
tar_obj = s3_client.get_object(Bucket=bucket, Key=tar_key)
tar_bytes = io.BytesIO(tar_obj['Body'].read())

extracted = {}  # filename -> bytes
with tarfile.open(fileobj=tar_bytes, mode='r:gz') as tar:
    for member in tar.getmembers():
        name = member.name.lstrip('./')
        f = tar.extractfile(member)
        if f:
            extracted[name] = f.read()

print(f"Extracted {len(extracted)} files:")
for k in sorted(extracted):
    print(f"  {k}")

# ── 1. Display pre-generated PNGs ────────────────────────────────────────────
png_found = False
for plot_name in ['plots/ttft_timeline.png', 'plots/ttft_over_time.png']:
    data = extracted.get(plot_name)
    if data:
        display(Markdown(f"**{plot_name}**"))
        display(Image(data=data, format='png'))
        png_found = True

# ── 2. Fallback: generate from profile_export.jsonl ──────────────────────────
if not png_found:
    print("Pre-generated PNGs not found — generating from profile_export.jsonl...")
    jsonl_data = extracted.get('profile_export.jsonl', b'')
    records = [json.loads(line) for line in jsonl_data.decode('utf-8').splitlines() if line.strip()]

    ttft_ms      = [r['metrics']['time_to_first_token']['value'] for r in records]
    latency_ms   = [r['metrics']['request_latency']['value'] for r in records]
    req_start_ns = [r['metadata']['request_start_ns'] for r in records]

    t0        = min(req_start_ns)
    elapsed_s = [(t - t0) / 1e9 for t in req_start_ns]
    req_idx   = list(range(1, len(records) + 1))
    p50 = np.percentile(ttft_ms, 50)
    p90 = np.percentile(ttft_ms, 90)
    p99 = np.percentile(ttft_ms, 99)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.scatter(req_idx, ttft_ms, color='#0072B2', s=60, zorder=3, label='TTFT per request')
    ax.plot(req_idx, ttft_ms, color='#0072B2', linewidth=1, alpha=0.4)
    ax.axhline(p50, color='#009E73', linestyle='--', linewidth=1.2, label=f'p50 = {p50:.1f} ms')
    ax.axhline(p90, color='#E69F00', linestyle='--', linewidth=1.2, label=f'p90 = {p90:.1f} ms')
    ax.axhline(p99, color='#D55E00', linestyle='--', linewidth=1.2, label=f'p99 = {p99:.1f} ms')
    ax.set_xlabel('Request #', fontsize=11)
    ax.set_ylabel('Time to First Token (ms)', fontsize=11)
    ax.set_title('TTFT per Request — Timeline Order', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 5))
    sc = ax.scatter(elapsed_s, ttft_ms, c=latency_ms, cmap='plasma',
                    s=70, zorder=3, edgecolors='white', linewidths=0.4)
    plt.colorbar(sc, ax=ax).set_label('Request Latency (ms)', fontsize=9)
    ax.axhline(p50, color='#009E73', linestyle='--', linewidth=1.2, label=f'p50 = {p50:.1f} ms')
    ax.axhline(p90, color='#E69F00', linestyle='--', linewidth=1.2, label=f'p90 = {p90:.1f} ms')
    ax.set_xlabel('Elapsed Time (s)', fontsize=11)
    ax.set_ylabel('Time to First Token (ms)', fontsize=11)
    ax.set_title('TTFT Over Time (colored by request latency)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

## Step 13 — Cleanup benchmark resources

Delete the benchmark job and workload config. A job must be in a terminal state before it can be deleted.

In [ ]:
# Delete a completed/failed/stopped job
client.delete_ai_benchmark_job(AIBenchmarkJobName=job_name)

In [ ]:
# Delete the workload config
client.delete_ai_workload_config(AIWorkloadConfigName=config_name)